In [1]:
import pandas as pd


df = pd.read_parquet('../Imdb_Movie_Dataset.parquet')
df_aux = df.copy()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
import xgboost as xgb
import numpy as np
import pandas as pd
import pickle
import gzip
import gc

df_aux['revenue'] = pd.to_numeric(df_aux['revenue'], errors='coerce')
df_aux['runtime'] = pd.to_numeric(df_aux['runtime'], errors='coerce')
df_aux['budget'] = pd.to_numeric(df_aux['budget'], errors='coerce')
df_aux['popularity'] = pd.to_numeric(df_aux['popularity'], errors='coerce')

df_with_runtime = df_aux[
    df_aux['runtime'].notna() & 
    (df_aux['runtime'] >= 2) & 
    (df_aux['runtime'] <= 250)
].copy()

df_without_runtime = df_aux[
    df_aux['runtime'].isna() | 
    (df_aux['runtime'] == 0) | 
    (df_aux['runtime'] < 2) | 
    (df_aux['runtime'] > 250)
].copy()

df_with_runtime['release_year'] = pd.to_datetime(df_with_runtime['release_date'], errors='coerce').dt.year
df_with_runtime['release_decade'] = (df_with_runtime['release_year'] // 10) * 10

df_with_runtime['overview_len'] = df_with_runtime['overview'].astype(str).str.len()
df_with_runtime['tagline_len'] = df_with_runtime['tagline'].astype(str).str.len()

df_with_runtime['keywords'] = df_with_runtime['keywords'].astype(str).fillna('')
df_with_runtime['is_short_keyword'] = df_with_runtime['keywords'].str.contains('short', case=False, regex=False).astype('int8')

df_with_runtime['has_budget'] = (df_with_runtime['budget'] > 0).astype('int8')
df_with_runtime['has_revenue'] = (df_with_runtime['revenue'] > 0).astype('int8')

df_with_runtime['genres'] = df_with_runtime['genres'].astype(str).fillna('')
df_with_runtime['main_genre'] = df_with_runtime['genres'].str.split(', ').str[0]

global_mean = df_with_runtime['runtime'].mean()
genre_stats = df_with_runtime.groupby('main_genre')['runtime'].agg(['mean', 'count'])

smoothing_g = 15
df_with_runtime['genre_mean'] = df_with_runtime['main_genre'].map(
    lambda x: (genre_stats.loc[x, 'mean'] * genre_stats.loc[x, 'count'] + global_mean * smoothing_g) / (genre_stats.loc[x, 'count'] + smoothing_g) if x in genre_stats.index else global_mean
)

genre_decade_stats = df_with_runtime.groupby(['main_genre', 'release_decade'])['runtime'].agg(['mean', 'count'])
smoothing_gd = 20
def get_smooth_genre_decade(row):
    key = (row['main_genre'], row['release_decade'])
    if key in genre_decade_stats.index:
        stat = genre_decade_stats.loc[key]
        return (stat['mean'] * stat['count'] + row['genre_mean'] * smoothing_gd) / (stat['count'] + smoothing_gd)
    return row['genre_mean']

df_with_runtime['genre_decade_mean'] = df_with_runtime.apply(get_smooth_genre_decade, axis=1)

df_with_runtime['budget_revenue_ratio'] = df_with_runtime['budget'] / (df_with_runtime['revenue'] + 1)
df_with_runtime['budget_to_popularity'] = df_with_runtime['budget'] / (df_with_runtime['popularity'] + 1)
df_with_runtime['popularity_to_votes'] = df_with_runtime['popularity'] / (df_with_runtime['vote_count'] + 1)
df_with_runtime['vote_score_interact'] = df_with_runtime['vote_average'] * np.log1p(df_with_runtime['vote_count'])
df_with_runtime['pop_genre_interact'] = df_with_runtime['popularity'] * df_with_runtime['genre_mean']
df_with_runtime['votes_per_year'] = df_with_runtime['vote_count'] / (2027 - df_with_runtime['release_year'] + 1)
df_with_runtime['is_en'] = (df_with_runtime['original_language'] == 'en').astype('int8')

numerical_features = [
    'vote_average', 'vote_count', 'release_year', 'release_decade', 'budget', 
    'popularity', 'is_short_keyword', 'overview_len', 'tagline_len', 
    'genre_mean', 'genre_decade_mean', 'budget_revenue_ratio', 
    'budget_to_popularity', 'popularity_to_votes', 'vote_score_interact',
    'has_budget', 'has_revenue', 'is_en', 'pop_genre_interact', 'votes_per_year'
]
multi_value_categorical_features = ['genres', 'production_companies', 'production_countries']
single_value_categorical_features = []

all_base_features = numerical_features + multi_value_categorical_features + single_value_categorical_features + ['main_genre']

df_train = df_with_runtime[all_base_features + ['runtime']].copy()
df_train.dropna(subset=['vote_average', 'vote_count', 'release_year', 'budget', 'popularity'], inplace=True)

top_n_categories = 50
cols_to_drop = ['main_genre']
new_features_dict = {}

for col in multi_value_categorical_features:
    df_train[col] = df_train[col].astype(str).fillna('')
    item_counts = df_train[col].str.split(', ').explode().str.strip().value_counts()
    top_items = item_counts[item_counts.index != ''].head(top_n_categories).index.tolist()
    
    for item_name in top_items:
        new_features_dict[f'{col}_{item_name}'] = df_train[col].str.contains(item_name, regex=False, na=False).astype('int8')
    
    cols_to_drop.append(col)

df_new_features = pd.DataFrame(new_features_dict, index=df_train.index)
df_train = pd.concat([df_train, df_new_features], axis=1)
df_train.drop(columns=cols_to_drop, inplace=True, errors='ignore')

df_train = pd.get_dummies(df_train, columns=[col for col in single_value_categorical_features], drop_first=True)

df_shorts = df_train[df_train['runtime'] < 40].copy()
df_features_shorts = df_shorts.drop('runtime', axis=1)
labels_shorts = np.log1p(df_shorts['runtime'])

df_features_longs = df_train[df_train['runtime'] >= 40].copy()
labels_longs = np.log1p(df_features_longs['runtime'])
df_features_longs.drop('runtime', axis=1, inplace=True)

features_for_prediction_final = df_features_longs.columns.tolist()

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(df_features_shorts, labels_shorts, test_size=0.2, random_state=42)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(df_features_longs, labels_longs, test_size=0.2, random_state=42)

del df_with_runtime, df_without_runtime, df_train, df_shorts, df_features_shorts, df_features_longs
del labels_shorts, labels_longs, df_new_features, new_features_dict
gc.collect()

model_shorts_runtime = xgb.XGBRegressor(
    n_estimators=600, learning_rate=0.03, max_depth=6, min_child_weight=2,
    subsample=0.8, colsample_bytree=0.7, random_state=42, n_jobs=-1
)
model_shorts_runtime.fit(X_train_s, y_train_s)

model_longs_runtime = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.02, max_depth=8, min_child_weight=4,
    subsample=0.8, colsample_bytree=0.6, gamma=0.5, random_state=42, n_jobs=-1
)
model_longs_runtime.fit(X_train_l, y_train_l)

y_pred_log_s = model_shorts_runtime.predict(X_test_s)
y_pred_log_l = model_longs_runtime.predict(X_test_l)

y_test_orig_s = np.expm1(y_test_s)
y_pred_orig_s = np.expm1(y_pred_log_s)

y_test_orig_l = np.expm1(y_test_l)
y_pred_orig_l = np.expm1(y_pred_log_l)

y_test_all = np.concatenate([y_test_orig_s, y_test_orig_l])
y_pred_all = np.concatenate([y_pred_orig_s, y_pred_orig_l])

r2 = r2_score(y_test_all, y_pred_all)
rmse = np.sqrt(mean_squared_error(y_test_all, y_pred_all))
mae = mean_absolute_error(y_test_all, y_pred_all)
mape = mean_absolute_percentage_error(y_test_all, y_pred_all)

print(f"\n--- Avaliação Combinada do Pipeline Especialista ---")
print(f"R² Score Global: {r2:.4f}")
print(f"RMSE Global: {rmse:,.2f} minutos")
print(f"MAE Global: {mae:,.2f} minutos")
print(f"MAPE Global: {mape * 100:.2f}%")

with gzip.open('models/xgb_shorts_runtime_model.pkl.gz', 'wb') as f:
    pickle.dump(model_shorts_runtime, f)

with gzip.open('models/xgb_longs_runtime_model.pkl.gz', 'wb') as f:
    pickle.dump(model_longs_runtime, f)
    
with open('models/features_runtime_model.pkl', 'wb') as f:
    pickle.dump(features_for_prediction_final, f)


--- Avaliação Combinada do Pipeline Especialista ---
R² Score Global: 0.7908
RMSE Global: 19.98 minutos
MAE Global: 12.29 minutos
MAPE Global: 34.17%
